In [1]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
import os

# ======================== Enhanced Core Components ========================

# 1. Data Preparation & SRL Processing
class SRLProcessor:
    def __init__(self):
        self.role_tags = {
            'V': 'verb', 'ARG0': 'subject', 'ARG1': 'object',
            'ARG2': 'indirect-object', 'ARGM-MNR': 'manner'
        }
    
    def process_srl(self, srl_data):
        """Process SRL data and add semantic role labels to text"""
        augmented = []
        for sent_info in srl_data:
            if 'srl_raw' not in sent_info or 'words' not in sent_info['srl_raw']:
                continue  # Skip malformed entries
                
            words = sent_info['srl_raw']['words']
            tags = [[] for _ in words]
            
            # Process each verb and its tags
            for verb_info in sent_info['srl_raw'].get('verbs', []):
                if 'tags' not in verb_info:
                    continue
                    
                current_tags = verb_info['tags']
                for idx, tag in enumerate(current_tags):
                    if idx >= len(tags):  # Prevent index error
                        break
                    if tag != 'O' and '-' in tag:
                        role = tag.split('-')[1]
                        if role in self.role_tags:  # Only add if it's in our defined roles
                            tags[idx].append(role)
            
            # Build the tagged sentence
            tagged_sentence = []
            for word, roles in zip(words, tags):
                for role in roles:
                    if role in self.role_tags:
                        tagged_sentence.append(f"[{self.role_tags[role]}]")
                tagged_sentence.append(word)
                for role in reversed(roles):
                    if role in self.role_tags:
                        tagged_sentence.append(f"[/{self.role_tags[role]}]")
            
            augmented.append(' '.join(tagged_sentence))
        
        return ' '.join(augmented)

def load_srl_dataset(json_path):
    """Load dataset from JSON and apply SRL processing"""
    try:
        with open(json_path) as f:
            data = json.load(f).get('samples', [])
    except (json.JSONDecodeError, FileNotFoundError) as e:
        print(f"Error loading JSON data: {e}")
        return pd.DataFrame()
    
    processor = SRLProcessor()
    samples = []
    
    for sample in tqdm(data, desc="Processing SRL data"):
        try:
            samples.append({
                'CVE_text': processor.process_srl(sample.get('CVE_srl', [])),
                'Technique_text': processor.process_srl(sample.get('Technique_srl', [])),
                'label': sample.get('label', 0),
                'role_score': sample.get('role_match_score', 0.0)
            })
        except Exception as e:
            print(f"Error processing sample: {e}")
            continue
    
    df = pd.DataFrame(samples)
    print(f"Loaded {len(df)} valid samples")
    return df
# 2. Dataset Class with Binary Labels
class FocalDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        # Normalize role weights to [0,1]
        if len(df) > 0:  # Check if df is not empty
            role_min = df['role_score'].min()
            role_max = df['role_score'].max()
            self.role_weights = (df['role_score'] - role_min) / (role_max - role_min + 1e-8)
            
            # Convert to -1/1 labels for compatibility with loss functions
            self.labels = 2 * df['label'].values - 1
        else:
            self.role_weights = pd.Series()
            self.labels = np.array([])

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Tokenize CVE text
        cve_encodings = self.tokenizer(
            row['CVE_text'], 
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Tokenize Technique text
        tech_encodings = self.tokenizer(
            row['Technique_text'],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'cve_input_ids': cve_encodings['input_ids'].squeeze(),
            'cve_attention_mask': cve_encodings['attention_mask'].squeeze(),
            'tech_input_ids': tech_encodings['input_ids'].squeeze(),
            'tech_attention_mask': tech_encodings['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float),
            'role_weights': torch.tensor(self.role_weights.iloc[idx], dtype=torch.float),
            'CVE_text': row['CVE_text'],
            'Technique_text': row['Technique_text']
        }


# 3. Focal Loss Implementation
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        """
        Focal Loss for binary classification.
        
        Args:
            gamma: Focusing parameter that reduces the relative loss for well-classified examples
            alpha: Weighting factor to balance positive/negative examples
        """
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
    
    def forward(self, outputs, labels, weights=None):
        """
        Calculate focal loss for binary classification with -1/1 labels
        
        Args:
            outputs: Raw model outputs (logits)
            labels: Binary labels in -1/1 format
            weights: Optional sample weights
        """
        # Convert -1/1 labels to 0/1 for BCE calculation
        targets = (labels + 1) / 2
        
        # Apply sigmoid to get probabilities
        probs = torch.sigmoid(outputs)
        
        # Calculate BCE loss
        bce_loss = F.binary_cross_entropy_with_logits(outputs, targets, reduction='none')
        
        # Calculate focal weights
        p_t = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        focal_weight = alpha_t * (1 - p_t).pow(self.gamma)
        
        # Apply focal weights to the BCE loss
        focal_loss = focal_weight * bce_loss
        
        # Apply optional sample weights
        if weights is not None:
            focal_loss = focal_loss * (1 + weights)
        
        return focal_loss.mean()

# 4. Enhanced Model Architecture with Focal and Contrastive Learning
class EnhancedFocalSRLModel(nn.Module):
    def __init__(self, model_name="bert-base-uncased", hidden_size=768):
        super().__init__()
        try:
            self.bert = AutoModel.from_pretrained(model_name)
        except Exception as e:
            print(f"Error loading pretrained model: {e}")
            raise RuntimeError("Failed to initialize the model")
        
        # Projection layer remains unchanged
        self.srl_proj = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.LayerNorm(256)
        )
        
        # Classifier using aggregation of embeddings and interaction features
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 1)
        )

    def soft_align_attention(self, a, b, mask_a, mask_b):
        """
        Compute soft-alignment between two sequences:
          a: [batch, seq_len_a, hidden]
          b: [batch, seq_len_b, hidden]
          mask_a: [batch, seq_len_a]
          mask_b: [batch, seq_len_b]
        Returns:
          aligned_a: weighted sum of b for each token in a
          aligned_b: weighted sum of a for each token in b
        """
        # Compute similarity matrix [B, L_a, L_b]
        similarity = torch.bmm(a, b.transpose(1, 2))
        
        # Create proper broadcasting dimensions for masks
        batch_size = mask_b.size(0)
        seq_len_a = similarity.size(1)
        seq_len_b = similarity.size(2)
        
        # Masking the padded tokens in b for computing attention on a's tokens
        mask_b_exp = mask_b.unsqueeze(1).expand(batch_size, seq_len_a, seq_len_b)
        attn_weights_a = F.softmax(similarity.masked_fill(mask_b_exp == 0, -1e9), dim=2)
        
        # Similarly, compute attention weights for b (using a's mask)
        mask_a_exp = mask_a.unsqueeze(2).expand(batch_size, seq_len_a, seq_len_b)
        attn_weights_b = F.softmax(similarity.transpose(1,2).masked_fill(mask_a_exp == 0, -1e9), dim=2)
        
        aligned_a = torch.bmm(attn_weights_a, b)  # [B, L_a, hidden]
        aligned_b = torch.bmm(attn_weights_b, a)  # [B, L_b, hidden]
        return aligned_a, aligned_b

    def pooling(self, token_embeddings, mask):
        """
        Apply masked average pooling to token embeddings.
          token_embeddings: [B, L, hidden]
          mask: [B, L] with 1 for valid tokens and 0 for padding.
        Returns:
          pooled embedding [B, hidden]
        """
        mask = mask.unsqueeze(2).float()  # [B, L, 1]
        summed = torch.sum(token_embeddings * mask, dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        
        # Add safety check for cases where mask is all zeros
        valid_counts = (counts > 1e-8).float()
        return (summed / counts) * valid_counts

    def forward(self, cve_input, tech_input, labels=None):
        # Encode inputs
        cve_outputs = self.bert(**cve_input)       # shape: [B, L_cve, hidden_size]
        tech_outputs = self.bert(**tech_input)     # shape: [B, L_tech, hidden_size]
        
        cve_seq = cve_outputs.last_hidden_state
        tech_seq = tech_outputs.last_hidden_state
        
        # Get the attention masks from inputs
        cve_mask = cve_input['attention_mask']     # [B, L_cve]
        tech_mask = tech_input['attention_mask']   # [B, L_tech]
        
        # -----------------------
        # Add: Soft Align Attention
        # -----------------------
        aligned_cve, aligned_tech = self.soft_align_attention(cve_seq, tech_seq, cve_mask, tech_mask)
        
        # Combine the original sequence with the aligned one (e.g., by averaging)
        cve_combined = (cve_seq + aligned_cve) / 2.0
        tech_combined = (tech_seq + aligned_tech) / 2.0
        
        # -----------------------
        # Add: Pooling over the token dimension
        # -----------------------
        cve_pooled = self.pooling(cve_combined, cve_mask)   # [B, hidden_size]
        tech_pooled = self.pooling(tech_combined, tech_mask)  # [B, hidden_size]
        
        # -----------------------
        # Apply projection to lower-dimensional embeddings
        # -----------------------
        cve_emb = self.srl_proj(cve_pooled)   # [B, 256]
        tech_emb = self.srl_proj(tech_pooled)   # [B, 256]
        
        # -----------------------
        # Aggregating features from both branches
        # -----------------------
        diff = torch.abs(cve_emb - tech_emb)
        prod = cve_emb * tech_emb
        combined_features = torch.cat([cve_emb, tech_emb, diff, prod], dim=1)  # [B, 256*4]
        
        # Compute classifier output (similarity score or decision)
        classifier_out = self.classifier(combined_features).squeeze()
        
        return classifier_out

    def get_embeddings(self, input_dict):
        # Compute embeddings for inference (using pooling, projection)
        with torch.no_grad():
            outputs = self.bert(**input_dict)
            pooled = self.pooling(outputs.last_hidden_state, input_dict['attention_mask'])
            return self.srl_proj(pooled).cpu().numpy()

# 5. Data Splitting with Stratification (unchanged)
def get_splits_for_model(df, test_size=0.15, val_size=0.15, random_state=42):
    """Split data into train/val/test with stratification"""
    if df.empty:
        raise ValueError("DataFrame is empty, cannot split")
        
    # First split off the test set
    train_val_df, test_df = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df['label']
    )
    
    # Then split the remaining data into train and validation
    relative_val_size = val_size / (1 - test_size)
    train_df, val_df = train_test_split(
        train_val_df, 
        test_size=relative_val_size, 
        random_state=random_state,
        stratify=train_val_df['label']
    )
    
    print(f"Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")
    print(f"Train label distribution: {train_df['label'].value_counts().to_dict()}")
    print(f"Val label distribution: {val_df['label'].value_counts().to_dict()}")
    print(f"Test label distribution: {test_df['label'].value_counts().to_dict()}")
    
    return train_df, val_df, test_df

# 6. Enhanced Training with Focal and Contrastive Loss
def train_focal_model(train_df, val_df=None, epochs=10, batch_size=16, 
                      patience=3, lr=2e-5):
    """Train model with MSE loss instead of focal loss."""
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    model = EnhancedFocalSRLModel().to(device)
    train_dataset = FocalDataset(train_df, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    if val_df is not None:
        val_dataset = FocalDataset(val_df, tokenizer)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)
    else:
        val_loader = None
    
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    mse_criterion = nn.L1Loss()
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_correct = train_total = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Training"):
            cve_input = {
                'input_ids': batch['cve_input_ids'].to(device),
                'attention_mask': batch['cve_attention_mask'].to(device)
            }
            tech_input = {
                'input_ids': batch['tech_input_ids'].to(device),
                'attention_mask': batch['tech_attention_mask'].to(device)
            }
            labels = batch['labels'].to(device).float()
            # If using weights, incorporate them appropriately
            weights = batch['role_weights'].to(device)

            optimizer.zero_grad()
            outputs = model(cve_input, tech_input)
            
            mse_loss = mse_criterion(outputs, labels)
            total_loss = mse_loss
            
            total_loss.backward()
            optimizer.step()
            
            train_loss += total_loss.item()
            
            # Assuming labels are 0/1, adjust prediction conversion accordingly.
            preds = (torch.sigmoid(outputs) > 0.5).float()
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
        
        avg_train_loss = train_loss / len(train_loader)
        train_acc = train_correct / train_total
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f}")
        
        if val_loader is not None:
            val_acc, val_loss = validate_model(model, val_loader, mse_criterion, device)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)
            
            print(f"Epoch {epoch+1}/{epochs} - Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                torch.save(model.state_dict(), "best_model.pth")
                print(f"Saved new best model with validation accuracy: {best_val_acc:.4f}")
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    model.load_state_dict(torch.load("best_model.pth"))
                    break
    
    if val_loader is None or patience_counter < patience:
        torch.save(model.state_dict(), "final_model.pth")
    
    plot_training_history(history)
    return model

def validate_model(model, val_loader, criterion, device):
    """Validate model on validation set."""
    model.eval()
    val_loss = 0
    val_correct = val_total = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            cve_input = {
                'input_ids': batch['cve_input_ids'].to(device),
                'attention_mask': batch['cve_attention_mask'].to(device)
            }
            tech_input = {
                'input_ids': batch['tech_input_ids'].to(device),
                'attention_mask': batch['tech_attention_mask'].to(device)
            }
            labels = batch['labels'].to(device).float()
            # Weights are unused here unless needed for loss calculation.
            weights = batch['role_weights'].to(device)
            
            outputs = model(cve_input, tech_input)
            
            # Use passed criterion
            total_loss = criterion(outputs, labels)
            val_loss += total_loss.item()
            
            # Adjust this prediction conversion if labels are not 0/1.
            preds = (torch.sigmoid(outputs) > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    
    return val_correct / val_total, val_loss / len(val_loader)


# 8. Helper functions (unchanged)
def plot_training_history(history):
    """Plot training and validation metrics"""
    plt.figure(figsize=(12, 5))
    
    # Plot loss
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    if 'val_loss' in history and history['val_loss']:
        plt.plot(history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Loss Curves')
    
    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    if 'val_acc' in history and history['val_acc']:
        plt.plot(history['val_acc'], label='Val Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Accuracy Curves')
    
    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.close()

# 9. Updated evaluation function for sigmoid outputs
def evaluate_model(model, test_loader, device):
    """Evaluate model and output detailed results on test set"""
    model.eval()
    
    # For metrics
    all_preds = []
    all_true = []
    all_outputs = []
    test_correct = test_total = 0
    
    # Test sample storage for detailed analysis
    test_samples = {
        'cve_text': [],
        'tech_text': [],
        'true_label': [],
        'predicted_label': [],
        'confidence_score': []
    }
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating on test set"):
            # Get batch data
            cve_input = {
                'input_ids': batch['cve_input_ids'].to(device),
                'attention_mask': batch['cve_attention_mask'].to(device)
            }
            
            tech_input = {
                'input_ids': batch['tech_input_ids'].to(device),
                'attention_mask': batch['tech_attention_mask'].to(device)
            }
            
            labels = batch['labels'].to(device)
            
            # Forward pass
            outputs = model(cve_input, tech_input)
            
            # Calculate predictions using sigmoid
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float() * 2 - 1  # Convert to -1/1
            
            # Calculate accuracy
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
            all_outputs.extend(probs.cpu().numpy())  # Store probabilities
            
            # Store samples for analysis
            for i in range(len(batch['cve_input_ids'])):
                test_samples['cve_text'].append(batch['CVE_text'][i])
                test_samples['tech_text'].append(batch['Technique_text'][i])
                test_samples['true_label'].append(float(labels[i].cpu().numpy()))
                test_samples['predicted_label'].append(float(preds[i].cpu().numpy()))
                test_samples['confidence_score'].append(float(probs[i].cpu().numpy()))
    
    # Calculate metrics
    test_acc = test_correct / test_total
    print(f"Test Accuracy: {test_acc:.4f}")
    
    # Create a DataFrame for test results for easier analysis and output
    test_results_df = pd.DataFrame(test_samples)
    
    # Convert labels from -1/1 to 0/1 for readability
    test_results_df['true_label'] = (test_results_df['true_label'] + 1) / 2
    test_results_df['predicted_label'] = (test_results_df['predicted_label'] + 1) / 2
    
    # Add a column for correct/incorrect predictions
    test_results_df['correct'] = test_results_df['true_label'] == test_results_df['predicted_label']
    
    # Save detailed test results to CSV
    test_results_df.to_csv('test_results_detailed.csv', index=False)
    print(f"Saved detailed test results to 'test_results_detailed.csv'")
    
    # Classification report
    all_true_01 = [(label + 1) / 2 for label in all_true]  # Convert -1/1 to 0/1
    all_preds_01 = [(pred + 1) / 2 for pred in all_preds]  # Convert -1/1 to 0/1
    
    class_report = classification_report(all_true_01, all_preds_01, output_dict=True)
    print("\nClassification Report:")
    for label, metrics in class_report.items():
        if label in ['0.0', '1.0']:
            print(f"Class {label}: Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, F1: {metrics['f1-score']:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(all_true_01, all_preds_01)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig('confusion_matrix.png')
    plt.close()
    
    # Error analysis summary
    print("\n=== Error Analysis Summary ===")
    print(f"Total test samples: {len(test_results_df)}")
    print(f"Correct predictions: {len(test_results_df[test_results_df['correct']])} ({len(test_results_df[test_results_df['correct']])/len(test_results_df)*100:.2f}%)")
    print(f"Incorrect predictions: {len(test_results_df[~test_results_df['correct']])} ({len(test_results_df[~test_results_df['correct']])/len(test_results_df)*100:.2f}%)")
    
    # Analyze high confidence errors
    high_conf_errors = test_results_df[(~test_results_df['correct']) & 
                                       ((test_results_df['confidence_score'] > 0.8) | 
                                       (test_results_df['confidence_score'] < 0.2))]
    print(f"High confidence errors: {len(high_conf_errors)} ({len(high_conf_errors)/len(test_results_df)*100:.2f}% of all samples)")
    
    return test_acc, test_results_df
# 10. T-SNE Visualization of Embeddings
def visualize_embeddings(model, test_loader, device, n_samples=500):
    """Create T-SNE visualization of the learned embeddings"""
    model.eval()
    
    # Collect embeddings and labels
    cve_embeddings = []
    tech_embeddings = []
    true_labels = []
    match_status = []  # Whether the pair is a match or not
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Extracting embeddings"):
            cve_input = {
                'input_ids': batch['cve_input_ids'].to(device),
                'attention_mask': batch['cve_attention_mask'].to(device)
            }
            tech_input = {
                'input_ids': batch['tech_input_ids'].to(device),
                'attention_mask': batch['tech_attention_mask'].to(device)
            }
            labels = batch['labels'].numpy()
            
            # Get embeddings (using the model's embedding extraction method)
            cve_emb = model.get_embeddings(cve_input)
            tech_emb = model.get_embeddings(tech_input)
            
            cve_embeddings.extend(cve_emb)
            tech_embeddings.extend(tech_emb)
            true_labels.extend(labels)
            
            # Determine match status (1 for match, 0 for non-match)
            match_status.extend([(label + 1) / 2 for label in labels])
            
            if len(cve_embeddings) >= n_samples:
                break
    
    # Limit to n_samples
    cve_embeddings = np.array(cve_embeddings[:n_samples])
    tech_embeddings = np.array(tech_embeddings[:n_samples])
    true_labels = np.array(true_labels[:n_samples])
    match_status = np.array(match_status[:n_samples])
    
    # Combine all embeddings for T-SNE
    all_embeddings = np.vstack([cve_embeddings, tech_embeddings])
    all_types = ['CVE'] * len(cve_embeddings) + ['Technique'] * len(tech_embeddings)
    
    # Create labels for the plots - combines the source type and match status
    embedding_labels = []
    for i in range(len(cve_embeddings)):
        is_match = "Match" if match_status[i] == 1 else "Non-match"
        embedding_labels.extend([f"CVE ({is_match})", f"Technique ({is_match})"])
    
    # Apply T-SNE for dimensionality reduction
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    reduced_embeddings = tsne.fit_transform(all_embeddings)
    
    # Plot the embeddings
    plt.figure(figsize=(12, 10))
    
    # Create a color map for the different categories
    color_map = {
        "CVE (Match)": "green",
        "Technique (Match)": "limegreen",
        "CVE (Non-match)": "red",
        "Technique (Non-match)": "salmon"
    }
    
    # Plot each point with appropriate color
    for i, label in enumerate(embedding_labels):
        plt.scatter(
            reduced_embeddings[i, 0],
            reduced_embeddings[i, 1],
            c=color_map[label],
            label=label,
            alpha=0.7,
            s=50
        )
    
    # Remove duplicate labels for legend
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys(), loc='best')
    
    plt.title("t-SNE Visualization of CVE and Technique Embeddings")
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.grid(alpha=0.3)
    plt.savefig('embeddings_tsne.png')
    plt.close()
    
    print(f"Saved t-SNE visualization to 'embeddings_tsne.png'")
    
# 11. Main Function to Execute the Pipeline
def main():
    """Main function to run the entire pipeline"""
    # Load and process data
    print("Loading and processing SRL dataset...")
    data_path = "/kaggle/input/siamesedataset3/siamese_samples_with_srl (6).json"  # Path to your SRL processed dataset
    df = load_srl_dataset(data_path)
    
    if df.empty:
        print("Error: Failed to load dataset!")
        return
    
    # Split data
    print("\nSplitting data into train/val/test sets...")
    try:
        train_df, val_df, test_df = get_splits_for_model(df)
    except ValueError as e:
        print(f"Error: {e}")
        return
    
    # Initialize tokenizer
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    
    # Train model
    print("\nTraining model...")
    model = train_focal_model(
    train_df=train_df,
    val_df=val_df,
    epochs=10,
    batch_size=16,
    patience=3,
    lr=2e-5
    )

    
    # Prepare test dataset and loader
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_dataset = FocalDataset(test_df, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=16)
    
    # Evaluate the model
    print("\nEvaluating model on test set...")
    test_acc, test_results = evaluate_model(model, test_loader, device)
    
    # Visualize embeddings
    print("\nGenerating t-SNE visualization of embeddings...")
    visualize_embeddings(model, test_loader, device)
    
    print("\nModel training and evaluation complete!")
    print(f"Final test accuracy: {test_acc:.4f}")
    print("Results saved to CSV and visualizations saved as PNG files.")

if __name__ == "__main__":
    main()

Loading and processing SRL dataset...


Processing SRL data: 100%|██████████| 6759/6759 [00:02<00:00, 2793.75it/s]


Loaded 6759 valid samples

Splitting data into train/val/test sets...
Train: 4731, Validation: 1014, Test: 1014
Train label distribution: {0: 3581, 1: 1150}
Val label distribution: {0: 767, 1: 247}
Test label distribution: {0: 767, 1: 247}


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Epoch 10/10 - Val Loss: 0.2209, Val Acc: 0.1686

Evaluating model on test set...


Evaluating on test set: 100%|██████████| 64/64 [00:22<00:00,  2.90it/s]


Test Accuracy: 0.9043
Saved detailed test results to 'test_results_detailed.csv'

Classification Report:
Class 0.0: Precision: 0.9146, Recall: 0.9635, F1: 0.9384
Class 1.0: Precision: 0.8641, Recall: 0.7206, F1: 0.7859

=== Error Analysis Summary ===
Total test samples: 1014
Correct predictions: 917 (90.43%)
Incorrect predictions: 97 (9.57%)
High confidence errors: 0 (0.00% of all samples)

Generating t-SNE visualization of embeddings...


Extracting embeddings:  48%|████▊     | 31/64 [00:10<00:11,  2.85it/s]


Saved t-SNE visualization to 'embeddings_tsne.png'

Model training and evaluation complete!
Final test accuracy: 0.9043
Results saved to CSV and visualizations saved as PNG files.
